In [1]:
import os
from glob import glob
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import torch

from monai.transforms import (
    LoadImaged, 
    Compose, 
    EnsureChannelFirstd, 
    CropForegroundd, 
    RandCropByPosNegLabeld, 
    Orientationd, 
    Spacingd, 
    ScaleIntensityRangePercentilesd,
    Lambdad,
    SpatialPadd,
    ResizeWithPadOrCropd
)

c:\Users\User\anaconda3\envs\monai\Lib\site-packages\ignite\handlers\checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [2]:
# images_dir = "dummy/images/raw_data/volumes/COLONOG"
# masks_dir = "dummy/masks/raw_data/labels/COLONOG"

images_dir = "test/images"
masks_dir = "test/masks"

images = sorted(glob(os.path.join(images_dir, "*.nii.gz")))
masks = sorted(glob(os.path.join(masks_dir, "*_seg.nii.gz")))

pairs = list(zip(images, masks))
print(pairs[0])

('test/images\\1.3.6.1.4.1.9328.50.4.0001.nii.gz', 'test/masks\\1.3.6.1.4.1.9328.50.4.0001_seg.nii.gz')


In [3]:
def show_comparison(raw_images, raw_masks, processed_images, processed_masks, slice_idx = None):

    # คำนวณ index จาก processed_images (ที่เล็กกว่า)
    axial = processed_images.shape[3] // 2
    coronal = processed_images.shape[2] // 2
    sagittal = processed_images.shape[1] // 2

    fig, axes = plt.subplots(3, 4, figsize = (10, 10))

    # Raw images ใช้ index ของตัวเอง
    raw_axial = raw_images.shape[3] // 2
    raw_coronal = raw_images.shape[2] // 2
    raw_sagittal = raw_images.shape[1] // 2

    axes[0, 0].imshow(np.rot90(raw_images[0, :, :, raw_axial], k=-1), cmap="gray")
    axes[0, 0].set_title("Raw Image (Axial)")

    axes[0, 1].imshow(np.rot90(raw_masks[0, :, :, raw_axial], k=-1))
    axes[0, 1].set_title("Raw Mask (Axial)")

    axes[0, 2].imshow(np.rot90(processed_images[0, :, :, axial], k=-1), cmap="gray")
    axes[0, 2].set_title("Preprocessed Image (Axial)")

    axes[0, 3].imshow(np.rot90(processed_masks[0, :, :, axial], k=-1))
    axes[0, 3].set_title("Preprocessed Mask (Axial)")

    axes[1, 0].imshow(np.rot90(raw_images[0, :, raw_coronal, :], k=1), cmap="gray")
    axes[1, 0].set_title("Raw Image (Coronal)")

    axes[1, 1].imshow(np.rot90(raw_masks[0, :, raw_coronal, :], k=1))
    axes[1, 1].set_title("Raw Mask (Coronal)")

    axes[1, 2].imshow(np.rot90(processed_images[0, :, coronal, :], k=1), cmap="gray")
    axes[1, 2].set_title("Preprocessed Image (Coronal)")

    axes[1, 3].imshow(np.rot90(processed_masks[0, :, coronal, :], k=1))
    axes[1, 3].set_title("Preprocessed Mask (Coronal)")

    axes[2, 0].imshow(np.rot90(raw_images[0, raw_sagittal, :, :], k=1), cmap="gray")
    axes[2, 0].set_title("Raw Image (Sagittal)")

    axes[2, 1].imshow(np.rot90(raw_masks[0, raw_sagittal, :, :], k=1))
    axes[2, 1].set_title("Raw Mask (Sagittal)")

    axes[2, 2].imshow(np.rot90(processed_images[0, sagittal, :, :], k=1), cmap="gray")
    axes[2, 2].set_title("Preprocessed Image (Sagittal)")

    axes[2, 3].imshow(np.rot90(processed_masks[0, sagittal, :, :], k=1))
    axes[2, 3].set_title("Preprocessed Mask (Sagittal)")

    for ax in axes.flatten():
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [4]:
def build_lookup(config):
    lumbar_classes = config["data"]["lumbar_classes"]
    remapped_lumbar_classes = config["data"]["remapped_lumbar_classes"]

    max_label = max(lumbar_classes)
    lookup = np.zeros(max_label + 1, dtype = np.int16)

    for origin, remapped in zip(lumbar_classes, remapped_lumbar_classes):
        lookup[origin] = remapped

    return lookup


def label_remap(mask, config):
    lookup = build_lookup(config)

    if isinstance(mask, torch.Tensor):
        mask = mask.cpu().numpy()

    mask = mask.astype(np.int32)
    remapped = lookup[mask]

    return remapped

In [5]:
# กำหนด config
config = {
    "data": {
        "lumbar_classes": [0, 20, 21, 22, 23, 24, 25],
        "remapped_lumbar_classes": [0, 1, 2, 3, 4, 5, 6]
    }
}

from functools import partial
label_remap_fn = partial(label_remap, config=config)

In [ ]:
loader = Compose([
    LoadImaged(keys = ["image", "mask"], image_only = False),
    EnsureChannelFirstd(keys = ["image", "mask"])
])

preprocess = Compose([
    LoadImaged(keys = ["image", "mask"], 
               image_only = False),
    EnsureChannelFirstd(keys = ["image", "mask"]),
    ScaleIntensityRangePercentilesd(keys = ["image"],
                                    lower = 5,
                                    upper = 99,
                                    b_min = 0,
                                    b_max = 1,
                                    clip = True),
    Orientationd(keys = ["image", "mask"], 
                axcodes = "RAS"),
    Spacingd(keys = ["image", "mask"],
                pixdim = (1.0, 1.0, 1.0),
                mode = ("trilinear", "nearest")),
    Lambdad(keys = ["mask"], 
            func = label_remap_fn),
    CropForegroundd(keys = ["image", "mask"], 
                    source_key = "image", 
                    margin = 10)
])

for images_path, masks_path in pairs:
    raw = loader({"image": images_path, "mask": masks_path})
    preprocessd = preprocess({"image": images_path, "mask": masks_path})
    
    filename = os.path.basename(images_path)
    print("Filename: ", filename)

    print("Raw shape:", raw["image"].shape)
    print("Preprocessed shape:", preprocessd["image"].shape)
    
    print("\n=== Intensity Range ===")
    print("Raw - Min:", raw["image"].min().item(), "Max:", raw["image"].max().item())
    print("After Scale - Min:", preprocessd["image"].min().item(), "Max:", preprocessd["image"].max().item())

    print("\n=== Orientation ===")
    print(f"Raw Orientation:", nib.aff2axcodes(raw["image"].affine))
    print(f"After Orientationd Orientation:", nib.aff2axcodes(preprocessd["image"].affine))

    print("\n=== Affine ===")
    print(f"Raw Affine:\n", raw["image"].affine)
    print(f"After Orientationd Affine:\n", preprocessd["image"].affine)

    print("\n=== Spacing ===")
    print("Raw Spacing:", raw["image"].pixdim)
    print("After Spacingd:", preprocessd["image"].pixdim)

    print("\n=== Mask Unique Values ===")
    print("Raw Mask Unique Values:", np.unique(raw["mask"]))
    print("Preprocessed Mask Unique Values:", np.unique(preprocessd["mask"]))

    show_comparison(raw["image"], raw["mask"], preprocessd["image"], preprocessd["mask"])

monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


RuntimeError: applying transform <monai.transforms.utility.dictionary.Lambdad object at 0x0000021F2ACD8890>